# V38 Phase-Aware Reproducibility Audit

This notebook prevents a dangerous mistake: mixing the **historical V38 release** with the **current Kaggle data release**.

It does not train a model, infer hidden labels, alter predictions, or perform leaderboard tuning.

## Two different contracts

### Current Kaggle package
The currently supplied files have:
- train: 3292 rows
- validation: 500 rows
- test: 1000 rows
- first test IDs: `BICH_0003, BICH_0006, BICH_0007, BICH_0011`

### Historical V38-aligned assets
The supplied historical V2 probability file has:
- test: 1000 probability rows
- first IDs: `BICH_0003, BICH_0006, BICH_0015, BICH_0016`

These are not the same test set. Historical V38 predictions must never be row-mapped onto the current test set.


In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, hashlib, json, shutil

ROOT = Path(".")
LABELS = ['is_toxic', 'implicit_toxicity', 'sarcasm', 'contains_profanity', 'threatening', 'identity_attack', 'sexual_content']

EXPECTED_CURRENT = {
    "train_rows": 3292,
    "validation_rows": 500,
    "test_rows": 1000,
    "first_test_ids": ['BICH_0003', 'BICH_0006', 'BICH_0007', 'BICH_0011'],
    "validation_id_sha256": "c22008c12cf70d8071664bc50ba99bc0cfe3b76b8e5f0dbb1eba2e9abf217c6e",
    "test_id_sha256": "37c538884cd2c983f501f828fef29dd970c6fcd0c9683b717a09e941eb1d2a6b",
}

EXPECTED_HISTORICAL_V38 = {
    "v2_validation_rows": 500,
    "v2_test_rows": 1000,
    "first_v2_test_ids": ['BICH_0003', 'BICH_0006', 'BICH_0015', 'BICH_0016'],
    "v2_validation_id_sha256": "3c7828fb8810806265b04dee2c4eee1d2c0a5d11e436f194ce4ced2c5e409905",
    "v2_test_id_sha256": "5353b250a3ade5c44c4553795672fe430b269e3813cb3ab9cd4be3c42250ce69",
    "v38_positive_counts": [673,70,715,180,32,618,63],
}

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

def id_hash(ids):
    return hashlib.sha256(("\n".join(map(str,ids))+"\n").encode()).hexdigest()


In [ ]:
# Rename/upload your files to these canonical names before running.
train = pd.read_csv(ROOT/"train.csv", dtype={"ID":str})
valid = pd.read_csv(ROOT/"validation.csv", dtype={"ID":str})
test = pd.read_csv(ROOT/"test.csv", dtype={"ID":str})
sample = pd.read_csv(ROOT/"sample_submission.csv", dtype={"ID":str})

assert len(train)==EXPECTED_CURRENT["train_rows"]
assert len(valid)==EXPECTED_CURRENT["validation_rows"]
assert len(test)==EXPECTED_CURRENT["test_rows"]
assert test.ID.iloc[:4].tolist()==EXPECTED_CURRENT["first_test_ids"]
assert id_hash(valid.ID.tolist())==EXPECTED_CURRENT["validation_id_sha256"]
assert id_hash(test.ID.tolist())==EXPECTED_CURRENT["test_id_sha256"]
assert sample.ID.tolist()==test.ID.tolist()
assert sample.columns.tolist()==["ID"]+LABELS

print("PASS: files match the current Kaggle package represented by this audit.")


In [ ]:
# Optional historical assets. These should NOT align to current validation/test.
v2v_path = ROOT/"validation_ensemble_probabilities_v2.csv"
v2t_path = ROOT/"test_ensemble_probabilities_v2.csv"

if v2v_path.exists() and v2t_path.exists():
    v2v=pd.read_csv(v2v_path,dtype={"ID":str})
    v2t=pd.read_csv(v2t_path,dtype={"ID":str})
    assert len(v2v)==EXPECTED_HISTORICAL_V38["v2_validation_rows"]
    assert len(v2t)==EXPECTED_HISTORICAL_V38["v2_test_rows"]
    assert id_hash(v2v.ID.tolist())==EXPECTED_HISTORICAL_V38["v2_validation_id_sha256"]
    assert id_hash(v2t.ID.tolist())==EXPECTED_HISTORICAL_V38["v2_test_id_sha256"]

    val_overlap=len(set(valid.ID)&set(v2v.ID))
    test_overlap=len(set(test.ID)&set(v2t.ID))
    print("Historical/current validation-ID overlap:", val_overlap, "/ 500")
    print("Historical/current test-ID overlap:", test_overlap, "/ 1000")

    assert valid.ID.tolist()!=v2v.ID.tolist(), "Unexpected: historical V2 validation exactly matches current validation."
    assert test.ID.tolist()!=v2t.ID.tolist(), "Unexpected: historical V2 test exactly matches current test."

    print("PASS: historical assets are quarantined from the current release.")
else:
    print("Historical V2 probability files not present; current package verification is still valid.")


## Historical V38 submission check

If you have the exact historical `submission_v38_NO_PROFANITY_ABLATION.csv`, place it beside this notebook.

The notebook will verify it **against the historical V2 test IDs**, not against the current Kaggle test IDs.

A historical V38 CSV that passes this check is a valid frozen artifact for provenance/research, but it is **not a current-phase submission**.


In [ ]:
v38_path=ROOT/"submission_v38_NO_PROFANITY_ABLATION.csv"
if v38_path.exists():
    assert v2t_path.exists(), "Need historical test_ensemble_probabilities_v2.csv to verify V38 ID contract."
    v2t=pd.read_csv(v2t_path,dtype={"ID":str})
    v38=pd.read_csv(v38_path,dtype={"ID":str})
    assert v38.columns.tolist()==["ID"]+LABELS
    assert v38.ID.tolist()==v2t.ID.tolist(), "V38 does not match its historical test IDs."
    assert v38[LABELS].isin([0,1]).all().all()
    counts=v38[LABELS].sum().astype(int).tolist()
    assert counts==EXPECTED_HISTORICAL_V38["v38_positive_counts"], counts
    print("PASS: historical V38 frozen artifact verified.")
    print("Historical V38 SHA256:",sha256_file(v38_path))
else:
    print("No historical V38 CSV supplied; provenance can be audited but exact V38 bytes cannot be frozen here.")


## What must happen for the current Kaggle release

Do **not**:
- copy V38 predictions by row number;
- map old probabilities to new test rows;
- fill the 802 non-overlapping current test IDs using guessed labels;
- use public leaderboard behavior to reconstruct those labels.

To create a current submission, train/infer a system using the **current** train/validation/test release and generate predictions for its exact 1000 current test IDs.

That new submission should receive a new version name. It is not byte-identical V38.


In [ ]:
audit = {
    "current_release": {
        "train_rows": len(train),
        "validation_rows": len(valid),
        "test_rows": len(test),
        "first_test_ids": test.ID.iloc[:4].tolist(),
        "train_sha256": sha256_file(ROOT/"train.csv"),
        "validation_sha256": sha256_file(ROOT/"validation.csv"),
        "test_sha256": sha256_file(ROOT/"test.csv"),
        "sample_submission_sha256": sha256_file(ROOT/"sample_submission.csv"),
        "validation_id_sha256": id_hash(valid.ID.tolist()),
        "test_id_sha256": id_hash(test.ID.tolist()),
    },
    "historical_v38": {
        "directly_compatible_with_current_test": False,
        "expected_positive_counts": EXPECTED_HISTORICAL_V38["v38_positive_counts"],
        "known_missing_artifact": "exact fitted continuous implicit_toxicity meta-model",
        "tb_xlmr_repo": "aplycaebous/tb-XLM-R-fpt",
        "tb_xlmr_license_status": "unresolved: current public page exposes no model card/license",
        "bge_repo": "nahidstaq/bge-m3-bangla",
        "bge_license_status": "MIT on current public page",
    },
    "integrity": {
        "hidden_labels_used": False,
        "historical_predictions_mapped_to_current_test": False,
        "leaderboard_label_reconstruction": False,
    }
}
Path("v38_phase_aware_audit.json").write_text(json.dumps(audit,indent=2),encoding="utf-8")
print(json.dumps(audit,indent=2))
